In [39]:
import os
import numpy as np
import pandas as pd
import librosa
from spafe.fbanks.linear_fbanks import linear_filter_banks
from joblib import Parallel, delayed

In [40]:
TARGET_DURATION = 4.0
SR = 16000

N_FFT = 512
WIN_LENGTH = int(SR * 0.020)
HOP_LENGTH = int(SR * 0.010)
N_FILTERS = 20
N_COEFFS = 20
N_JOBS = -1

target_length = int(TARGET_DURATION * SR)
n_frames = target_length // HOP_LENGTH + 1

base_dataset_path = "../Data/DS_10283_3336/LA/"
output_path = "../Data/"
os.makedirs(output_path, exist_ok=True)

In [41]:
def read_protocol(path, dataset="2019"):
    if dataset == "2019":
        prot = pd.read_csv(path, sep=" ", header=None, names=["speaker", "filename", "dash", "attack", "key"])
    else:
        prot = pd.read_csv(path, sep=r"\s+", header=None, names=["speaker", "filename", "codec", "transmission", "attack", "key", "trim", "phase"])
    prot["is_bonafide"] = prot["key"] == "bonafide"
    return prot[["filename", "speaker", "attack", "is_bonafide"]].reset_index(drop=True)

In [42]:
LINEAR_FB, _ = linear_filter_banks(nfilts=N_FILTERS, nfft=N_FFT, fs=SR, low_freq=0, high_freq=SR / 2, scale="constant")
LINEAR_FB = LINEAR_FB.astype(np.float32)
MEL_FB = librosa.filters.mel(sr=SR, n_fft=N_FFT, n_mels=N_FILTERS, htk=True, norm=None)

In [43]:
def normalize_audio(path):
    y, sr = librosa.load(path, sr=None)

    if len(y) < target_length:
        reps = int(np.ceil(target_length / len(y)))
        y = np.tile(y, reps)

    return y[:target_length]

In [44]:
def cepstrum(power_spec, fb):
    return librosa.feature.mfcc(S=librosa.power_to_db(fb @ power_spec), n_mfcc=N_COEFFS)


def extract_audio_features(path):
    y = normalize_audio(path)
    spec = librosa.stft(y, n_fft=N_FFT, win_length=WIN_LENGTH, hop_length=HOP_LENGTH, window="hamming")
    power_spec = np.abs(spec) ** 2
    mfcc = cepstrum(power_spec, MEL_FB)
    lfcc = cepstrum(power_spec, LINEAR_FB)
    return np.stack([mfcc, lfcc]).astype(np.float16)

In [45]:
def process_split(split_name, flac_dir, protocol_path, dataset="2019"):
    prot = read_protocol(protocol_path, dataset)
    paths = [os.path.join(flac_dir, f"{name}.flac") for name in prot["filename"]]

    X = np.lib.format.open_memmap(
        f"{output_path}{split_name}_features.npy", mode="w+",
        dtype=np.float16, shape=(len(paths), 2, N_COEFFS, n_frames)
    )

    batch = 512
    for start in range(0, len(paths), batch):
        feats = Parallel(n_jobs=N_JOBS)(delayed(extract_audio_features)(p) for p in paths[start:start + batch])
        X[start:start + len(feats)] = np.stack(feats)
        print(f"\r{split_name}: {min(start + batch, len(paths))}/{len(paths)}", end="")

    X.flush()
    prot.to_csv(f"{output_path}{split_name}_meta.csv", index=False)
    print(f"\n{split_name}: {X.shape} | {os.path.getsize(f'{output_path}{split_name}_features.npy') / 1e9:.2f} GB | "
          f"bonafide={int(prot['is_bonafide'].sum())} spoof={int((~prot['is_bonafide']).sum())}")

In [46]:
process_split(
    "train",
    f"{base_dataset_path}ASVspoof2019_LA_train/flac",
    f"{base_dataset_path}ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt",
)

/home/assalim/Desktop/Master/A-Wavelet-Based-Convolutional-Neural-Network-for-Deepfake-Audio-Detection/venv/lib/python3.13/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/assalim/Desktop/Master/A-Wavelet-Based-Convolutional-Neural-Network-for-Deepfake-Audio-Detection/venv/lib/python3.13/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/assalim/Desktop/Master/A-Wavelet-Based-Convolutional-Neural-Network-for-Deepfake-Audio-Detec

train: 25380/25380
train: (25380, 2, 20, 401) | 0.81 GB | bonafide=2580 spoof=22800


In [47]:
process_split(
    "dev",
    f"{base_dataset_path}ASVspoof2019_LA_dev/flac",
    f"{base_dataset_path}ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt",
)

dev: 24844/24844
dev: (24844, 2, 20, 401) | 0.80 GB | bonafide=2548 spoof=22296


In [48]:
process_split(
    "eval",
    f"{base_dataset_path}ASVspoof2019_LA_eval/flac",
    f"{base_dataset_path}ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt",
)

eval: 71237/71237
eval: (71237, 2, 20, 401) | 2.29 GB | bonafide=7355 spoof=63882


In [49]:
process_split(
    "eval_2021",
    "../Data/ASVspoof2021_LA_eval/flac",
    "../Data/ASVspoof2021_LA_eval/trial_metadata.txt",
    dataset="2021"
)

eval_2021: 181566/181566
eval_2021: (181566, 2, 20, 401) | 5.82 GB | bonafide=18452 spoof=163114
